In [1]:
#میخوام ببینم مدل رندوم فورست به چه فیچرهایی اهمیت داده 
import pandas as pd
df = pd.read_csv("train.csv")

#پرکردن خانه های خالی از جنس استرینگ
df["PoolQC"] = df["PoolQC"].fillna("No Pool")
df["MiscFeature"] = df["MiscFeature"].fillna("None")
df["Alley"] = df["Alley"].fillna("No alley access")
df["Fence"] = df["Fence"].fillna("No Fence")
df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["FireplaceQu"] = df["FireplaceQu"].fillna("No Fireplace")
df["GarageQual"] = df["GarageQual"].fillna("No Garage")
df["GarageFinish"] = df["GarageFinish"].fillna("No Garage")
df["GarageType"] = df["GarageType"].fillna("No Garage")
df["GarageCond"] = df["GarageCond"].fillna("No Garage")
df["BsmtFinType2"] = df["BsmtFinType2"].fillna("No Basement")
df["BsmtExposure"] = df["BsmtExposure"].fillna("No Basement")
df["BsmtCond"] = df["BsmtCond"].fillna("No Basement")
df["BsmtQual"] = df["BsmtQual"].fillna("No Basement")
df["BsmtFinType1"] = df["BsmtFinType1"].fillna("No Basement")
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

#پرکردن خانه های خالی از جنس عدد
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

#آموزش و تست
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.8925099246778001

In [2]:
###############################################
#از اینجا جدیده

# حالا واسه اینکه بفهمیم مدل منطقی یاد گرفته یا نه و به فیچرهای بامعنی بیشتر اهمیت داده
# و همینطور برای بهتر کردن مدل که اگه فیچری خیلی ارزش نداره حذفش کنیم تا مدل رو ساده تر و سریعنر کنیم
#و همینطور برای Feature Engineering مهم است

importance = pd.Series( #Series به هر عدد یک برچسب (Index) وصل می‌کند.
    model.feature_importances_,  #کدام ستون‌ها برای پیش‌بینی قیمت در مدلی که انتخاب کردیم(رندوم فورست) مهم‌تر بودند؟
    index=X.columns #اسم ستون‌ها را به این عددها وصل می‌کنیم
)

importance.sort_values(ascending=False).head(10) 

OverallQual    0.557668
GrLivArea      0.120652
TotalBsmtSF    0.034901
2ndFlrSF       0.033897
BsmtFinSF1     0.028801
1stFlrSF       0.025387
LotArea        0.017154
GarageArea     0.015082
GarageCars     0.011828
YearBuilt      0.011605
dtype: float64

In [3]:
#OverallQual  0.557635 کیفیت کلی خانه
#یعنی حدود ۵۵٪ از تصمیم‌های Random Forest بر اساس این ویژگی بوده است.
#طبیعی است که کیفیت کلی مهم‌ترین عامل قیمت باشد.


#GrLivArea متراژ قسمت‌های قابل سکونت خانه
#هرچه خانه بزرگ‌تر باشد، معمولاً گران‌تر است.
#پس منطقی است دوم باشد

#TotalBsmtSFمتراژ کل زیرزمین
#زیرزمین بزرگ‌تر معمولاً ارزش خانه را بیشتر می‌کند.

#2ndFlrSF متراژ طبقه دوم
#اگر دو خانه زمین یکسانی داشته باشند، خانه‌ای که طبقه دوم دارد معمولاً فضای زندگی بیشتری دارد و ارزشش بالاتر است

#BsmtFinSF1 متراژ قسمت تکمیل‌شده (Finished) زیرزمین. از زیرزمین، چقدرش قابل استفاده و تکمیل شده است

#1stFlrSFمتراژ طبقه اول.

#LotArea مساحت زمین

#GarageArea متراژ گاراژ

#YearBuilt سال ساخت خانه جدیدتر گرونتر 

#GarageCars ظرفیت گاراژ از نظر تعداد خودرو


#شاید متوجه شده باشی که بعضی ویژگی‌ها به هم مرتبط هستند
#GrLivArea, 1stFlrSF , 2ndFlrSF
#TotalBsmtSF,BsmtFinSF1
#GarageArea, GarageCars
#این طبیعی است. این ویژگی‌ها اطلاعات مشابه ولی نه کاملاً یکسانی دارند.
#. Random Forest می‌تواند از ترکیب آن‌ها استفاده کند و تصمیم بگیرد هر کدام در چه شرایطی مفیدتر هستند.
#این مشاهده بعداً برای Feature Engineering و Feature Selection خیلی مهم می‌شود،
#، چون یاد می‌گیری گاهی دو ویژگی اطلاعات تقریباً تکراری دارند و گاهی هر کدام جنبه متفاوتی از یک موضوع را توصیف می‌کنند.


In [4]:
# از کجا بفهمیم ویژگی‌هایی که مدل مهم تشخیص داده منطقی هستند؟
# ابتدا باید آن‌ها را با دانش مسئله مقایسه کنیم. مثلاً در قیمت‌گذاری خانه انتظار داریم
# کیفیت ساخت، متراژ، سال ساخت و مساحت زمین از عوامل مهم باشند. اگر مدل نیز همین
# ویژگی‌ها را مهم بداند، یعنی احتمالاً الگوی درستی یاد گرفته است. علاوه بر این، می‌توان
# با حذف یک ویژگی و آموزش دوباره مدل بررسی کرد که آیا عملکرد مدل افت می‌کند یا خیر؛
# اگر حذف آن باعث افت عملکرد شود، یعنی آن ویژگی واقعاً برای مدل مفید بوده است.

In [5]:
# مهم بودن یک ویژگی در این مدل(رندوم فورست) به معنی وزن بیشتر نیست. این مدل مانند رگرسیون خطی
# برای هر ویژگی یک ضریب ثابت یاد نمی‌گیرد. بلکه هنگام ساخت درخت‌ها بارها بررسی
# می‌کند که استفاده از هر ویژگی چقدر خطای پیش‌بینی را کاهش می‌دهد. هر ویژگی که
# در مجموع باعث کاهش بیشتری در خطا شود، مقدار اهمیت بالاتری می‌گیرد. بنابراین
# اهمیت یک ویژگی یعنی میزان کمک آن به تصمیم‌گیری مدل، نه اینکه وزن یا ضریب
# بزرگ‌تری داشته باشد.